# 고혈압 예측 - CatBoost Optuna 하이퍼파라미터 튜닝

- 타겟: `고혈압유병` (0: 없음 / 1: 있음)
- 데이터: x1_preprocessed.csv
- 모델: CatBoost + Optuna TPE Sampler
- Threshold: **0.50 고정**
- 목표: **Recall ≥ 0.80 유지하면서 F1 최대화**
- 검증: Stratified 5-Fold CV

In [ ]:
import os
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold

optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")
matplotlib.rcParams["font.family"] = "DejaVu Sans"

INPUT_PATH = "/Users/Jiyeon/Desktop/final_project/ML/data/x1_preprocessed.csv"
NPY_DIR = "/Users/Jiyeon/Desktop/final_project/ML/outputs/oof"
RANDOM_STATE = 42
THRESHOLD = 0.50
N_TRIALS = 50

## 1. 데이터 로드 & 피처/타겟 분리

In [ ]:
df = pd.read_csv(INPUT_PATH)
TARGET = "고혈압유병"
DROP_COLS = ["고혈압유병", "당뇨유병", "이상지질혈증유병", "비만단계"]

data = df.dropna(subset=[TARGET]).copy()
X = data.drop(columns=DROP_COLS)
y = data[TARGET].astype(int)
neg, pos = (y == 0).sum(), (y == 1).sum()
ratio = neg / pos
print(f"샘플 수: {len(y)}  |  정상: {neg}  |  고혈압: {pos}")
print(f"class_weights: {{0: 1.0, 1: {ratio:.4f}}}")

## 2. Optuna Objective 정의

In [ ]:
def objective(trial):
    params = dict(
        iterations=trial.suggest_int("iterations", 300, 800),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        depth=trial.suggest_int("depth", 3, 8),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        bagging_temperature=trial.suggest_float("bagging_temperature", 0.0, 1.0),
        random_strength=trial.suggest_float("random_strength", 0.0, 1.0),
        border_count=trial.suggest_int("border_count", 32, 255),
        loss_function="Logloss",
        eval_metric="AUC",
        class_weights={0: 1.0, 1: ratio},
        early_stopping_rounds=50,
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
    )
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    scores = []
    for tr_idx, val_idx in cv.split(X, y):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        m = CatBoostClassifier(**params)
        m.fit(Pool(X_tr, y_tr), eval_set=Pool(X_val, y_val))
        proba = m.predict_proba(X_val)[:, 1]
        pred = (proba >= THRESHOLD).astype(int)
        recall = recall_score(y_val, pred)
        penalty = max(0, 0.8 - recall) * 2
        scores.append(f1_score(y_val, pred) - penalty)
    return np.mean(scores)

## 3. Optuna 최적화 실행

In [ ]:
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f"\n최적 trial: {study.best_trial.number}")
print(f"최적 score: {study.best_value:.4f}")
print("\n[최적 파라미터]")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

## 4. Optuna 최적화 과정 시각화

In [ ]:
trial_values = [t.value for t in study.trials]
best_so_far = np.maximum.accumulate(trial_values)

plt.figure(figsize=(10, 4))
plt.plot(trial_values, alpha=0.4, color="steelblue", label="Trial score")
plt.plot(best_so_far, color="tomato", lw=2, label="Best so far")
plt.axhline(y=0.6475, color="gray", linestyle="--", alpha=0.7, label="Baseline F1=0.6475")
plt.xlabel("Trial")
plt.ylabel("Score (F1 - penalty)")
plt.title("Optuna 최적화 과정 — 고혈압 CatBoost")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 최적 파라미터로 최종 학습 & 평가

In [ ]:
best_params = study.best_params.copy()
best_params.update(
    dict(
        loss_function="Logloss",
        eval_metric="AUC",
        class_weights={0: 1.0, 1: ratio},
        early_stopping_rounds=50,
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
    )
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_proba = np.zeros(len(y))
fold_scores = []

print("=" * 65)
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    model = CatBoostClassifier(**best_params)
    model.fit(Pool(X_tr, y_tr), eval_set=Pool(X_val, y_val))
    proba = model.predict_proba(X_val)[:, 1]
    oof_proba[val_idx] = proba
    pred = (proba >= THRESHOLD).astype(int)
    cm_f = confusion_matrix(y_val, pred)
    fold_scores.append(
        {
            "fold": fold,
            "auc": roc_auc_score(y_val, proba),
            "f1": f1_score(y_val, pred),
            "recall": recall_score(y_val, pred),
            "precision": precision_score(y_val, pred),
            "fp": int(cm_f[0, 1]),
            "best_iter": model.best_iteration_,
        }
    )
    print(
        f"  Fold {fold} | AUC: {fold_scores[-1]['auc']:.4f} | "
        f"Recall: {fold_scores[-1]['recall']:.4f} | "
        f"F1: {fold_scores[-1]['f1']:.4f} | "
        f"best_iter: {model.best_iteration_}"
    )

scores_df = pd.DataFrame(fold_scores)
print("=" * 65)
print(
    f"  평균   | AUC: {scores_df.auc.mean():.4f}±{scores_df.auc.std():.4f} "
    f"| Recall: {scores_df.recall.mean():.4f}±{scores_df.recall.std():.4f} "
    f"| F1: {scores_df.f1.mean():.4f}±{scores_df.f1.std():.4f}"
)

## OOF proba 저장 (.npy)

In [ ]:
os.makedirs(NPY_DIR, exist_ok=True)
npy_path = os.path.join(NPY_DIR, "oof_proba_HTN_catboost_optuna.npy")
oof_array = np.stack([oof_proba, y.values], axis=1)
np.save(npy_path, oof_array)
print(f"저장 완료 → {npy_path}")
loaded = np.load(npy_path)
print(f"로드 확인: shape={loaded.shape}, 일치={np.allclose(oof_array, loaded)}")

## 6. OOF 전체 성능 & 비교

In [ ]:
pred_oof = (oof_proba >= THRESHOLD).astype(int)
cm = confusion_matrix(y, pred_oof)
oof_auc = roc_auc_score(y, oof_proba)
oof_rec = recall_score(y, pred_oof)
oof_prec = precision_score(y, pred_oof)
oof_f1 = f1_score(y, pred_oof)
oof_acc = float((pred_oof == y).mean())

BASE = {"auc": 0.8582, "recall": 0.8310, "precision": 0.5304, "f1": 0.6475, "acc": 0.7556, "fp": 1236, "fn": 284}

print("=" * 55)
print(f"  {'지표':<12}  {'Baseline':>12}  {'Optuna':>10}  변화")
print("=" * 55)
for label, base_v, opt_v in [
    ("AUC-ROC", BASE["auc"], oof_auc),
    ("Recall", BASE["recall"], oof_rec),
    ("Precision", BASE["precision"], oof_prec),
    ("F1-score", BASE["f1"], oof_f1),
    ("Accuracy", BASE["acc"], oof_acc),
]:
    d = opt_v - base_v
    arrow = "▲" if d > 0 else ("▼" if d < 0 else "─")
    print(f"  {label:<12}  {base_v:>12.4f}  {opt_v:>10.4f}  {arrow} {abs(d):.4f}")
print(
    f"  {'FP':<12}  {BASE['fp']:>12}  {cm[0, 1]:>10}  {'▼' if cm[0, 1] < BASE['fp'] else '▲'} {abs(cm[0, 1] - BASE['fp'])}"
)
print(
    f"  {'FN':<12}  {BASE['fn']:>12}  {cm[1, 0]:>10}  {'▼' if cm[1, 0] < BASE['fn'] else '▲'} {abs(cm[1, 0] - BASE['fn'])}"
)
print("=" * 55)
print("\n[분류 리포트 — Optuna]")
print(classification_report(y, pred_oof, target_names=["정상(0)", "고혈압(1)"]))

## 7. Confusion Matrix 비교

In [ ]:
cm_base = np.array([[3117, 1236], [284, 1396]])
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, cm_data, title in zip(axes, [cm_base, cm], ["Baseline (threshold=0.50)", "Optuna (threshold=0.50)"]):
    ConfusionMatrixDisplay(cm_data, display_labels=["정상(0)", "고혈압(1)"]).plot(cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(title)
plt.suptitle("고혈압 CatBoost — Confusion Matrix 비교", fontsize=13)
plt.tight_layout()
plt.show()

## 8. Feature Importance Top 15 (마지막 fold)

In [ ]:
fi = pd.DataFrame({"feature": X.columns, "importance": model.get_feature_importance()}).sort_values(
    "importance", ascending=False
)

plt.figure(figsize=(8, 6))
fi.head(15).set_index("feature")["importance"][::-1].plot(kind="barh", color="steelblue")
plt.title("Feature Importance Top 15 — 고혈압 CatBoost Optuna")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()
print(fi.head(15).to_string(index=False))

## 9. DB 로그 저장

In [ ]:
import sys

sys.path.insert(0, "/Users/Jiyeon/Desktop/final_project/ML")
from model_logger import ModelLogger

logger = ModelLogger("/Users/Jiyeon/Desktop/final_project/ML/model_result.db")
run_id = logger.log_run(
    target_var="고혈압",
    model_name="CatBoost",
    stage="optuna",
    hyperparams={
        "learning_rate": best_params["learning_rate"],
        "depth": best_params["depth"],
        "n_estimators": best_params["iterations"],
        "class_weight": {0: 1.0, 1: round(ratio, 4)},
        "l2_leaf_reg": best_params["l2_leaf_reg"],
        "bagging_temperature": best_params["bagging_temperature"],
        "random_strength": best_params["random_strength"],
        "border_count": best_params["border_count"],
    },
    data_info={"feature_count": X.shape[1], "train_test_split": "5-Fold CV", "scaling_method": "None"},
    oof_metrics={
        "accuracy": oof_acc,
        "recall": oof_rec,
        "precision": oof_prec,
        "f1_score": oof_f1,
        "auc_roc": oof_auc,
        "cm": cm.tolist(),
    },
    fold_scores=scores_df.to_dict("records"),
    top_features=fi.set_index("feature")["importance"].head(15).to_dict(),
    note=f"Optuna {N_TRIALS} trials. threshold={THRESHOLD} 고정.",
)
print(f"저장 완료 → run_id: {run_id}")
print(logger.compare_runs().to_string(index=False))